In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib
import warnings

warnings.filterwarnings("ignore")

#### Обучите модель машинного обучения, которая дает наибольшую эффективность на обучающей выборке.

Лучшие результаты у меня показали CatBoostRegressor и XGBRegressor. Я возьму второй (так как он мне больше нравится).

In [28]:
df = pd.read_csv('D:/Maga/6th module/MLOps/ml/data/final_df.csv', index_col=0)

df

,brand,categories,colors,price,is_available,condition,is_sale,source,shipping,is_special_offer,is_returnable,manufacturer
1284,Forever Young,shoes,black,29.99,1,New,1,Overstock.com,No,1,0,RB-1904
1284,Forever Young,clothing,black,29.99,1,New,1,Overstock.com,No,1,0,RB-1904
1286,Forever Young,shoes,black,49.88,1,New,1,Overstock.com,No,1,0,RB-1904
1286,Forever Young,clothing,black,49.88,1,New,1,Overstock.com,No,1,0,RB-1904
7862,Hush Puppies,shoes,black,23.54,1,New,1,Overstock.com,No,1,0,H509362-635-75 M US
...,...,...,...,...,...,...,...,...,...,...,...,...
33797,Novica,clothing,red,39.99,1,New,1,Overstock.com,No,0,0,276629
33797,Novica,shoes,red,39.99,1,New,1,Overstock.com,No,0,0,276629
33798,Novica,accessories,red,35.99,1,New,0,Overstock.com,No,0,0,276629
33798,Novica,clothing,red,35.99,1,New,0,Overstock.com,No,0,0,276629


In [29]:
# Категориальные признаки:
# brand
# condition
# source
# shipping
# manufacturer
# category
# color

categorical_cols = df.dtypes[df.dtypes == 'object'].index
categorical_cols

Index(['brand', 'categories', 'colors', 'condition', 'source', 'shipping',
       'manufacturer'],
      dtype='object')

In [30]:
for col in categorical_cols:
    le = LabelEncoder()
    save_path = f'D:/Maga/6th module/MLOps/ml/data/encoders/{col}_classes.npy'
    le.classes_ = np.load(save_path, allow_pickle=True)
    df[col] = le.transform(df[col])
    
df

,brand,categories,colors,price,is_available,condition,is_sale,source,shipping,is_special_offer,is_returnable,manufacturer
1284,281,2,1,29.99,1,1,1,12,1,1,0,1831
1284,281,1,1,29.99,1,1,1,12,1,1,0,1831
1286,281,2,1,49.88,1,1,1,12,1,1,0,1831
1286,281,1,1,49.88,1,1,1,12,1,1,0,1831
7862,324,2,1,23.54,1,1,1,12,1,1,0,1355
...,...,...,...,...,...,...,...,...,...,...,...,...
33797,500,1,6,39.99,1,1,1,12,1,0,0,293
33797,500,2,6,39.99,1,1,1,12,1,0,0,293
33798,500,0,6,35.99,1,1,0,12,1,0,0,293
33798,500,1,6,35.99,1,1,0,12,1,0,0,293


Препроцессинг был выполнен при помощи сохраненных с прошлой лабы параметров/объектов.

In [31]:
# выделение столбцов-признаков и целевой переменной
X, y = df.drop(columns=['price']).to_numpy(), df['price'].to_numpy()

# разделение на обучающее и тестовое множества
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [32]:
scaler_filename = 'D:/Maga/6th module/MLOps/ml/data/scalers/y_scaler.save'
scaler = joblib.load(scaler_filename)

y_train = scaler.transform(y_train.reshape(-1, 1))

In [ ]:
# для удобной печати метрик

from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import r2_score as R2

def show_results():
    res_df = pd.DataFrame(results)
    res_df.set_index('model', inplace=True)
    pd.set_option('display.float_format', lambda x: '%.9f' % x)
    return res_df

def add_metric(title, rmse, mae, r2):
    results['model'].append(title)
    results['RMSE'].append(rmse)
    results['MAE'].append(mae)
    results['R2'].append(r2)

def calculate_metrics(title, y_train, predict_train, y_test, predict_test):
    print(title)
    print('RMSE')
    rmse = MSE(y_test, predict_test, squared=False)
    print('Train: ', MSE(y_train, predict_train, squared=False))
    print('Test: ', rmse)
    print('MAE')
    mae = MAE(y_test, predict_test)
    print('Train: ', MAE(y_train, predict_train))
    print('Test: ', mae)
    print('R2')
    r2 = R2(y_test, predict_test)
    print('Train: ', R2(y_train, predict_train))
    print('Test: ', r2)
    add_metric(title, rmse, mae, r2)
    print()

global indexes
global results

indexes = []
results = {'model': [], 'RMSE': [], 'MAE': [], 'R2': []}

In [34]:
model = xgb.XGBRegressor()
model.fit(X_train, y_train)

predict_train = model.predict(X_train)
predict_test_norm = model.predict(X_test)

predict_test = scaler.inverse_transform(predict_test_norm.reshape(-1, 1))

calculate_metrics('XGBRegressor', y_train, predict_train, y_test, predict_test)

XGBRegressor
RMSE
Train:  0.01897497912788199
Test:  38.09120182649593
MAE
Train:  0.008460671445452076
Test:  15.303332630677186
R2
Train:  0.7657834691261389
Test:  0.6926842510138067



#### Сериализуйте обученную модель используя такие библиотеки как joblib или pickle. Можно использовать любой другой метод записи объекта модели в файл.

In [35]:
model_filename = 'D:/Maga/6th module/MLOps/ml/data/models/xgb.save'
print(f'Saving model to {scaler_filename}...')
joblib.dump(model, model_filename)

Saving model to D:/Maga/6th module/MLOps/ml/data/scalers/y_scaler.save...


['D:/Maga/6th module/MLOps/ml/data/models/xgb.save']

In [37]:
# просто для теста попробую прочитать объект из файла

loaded_model = joblib.load(model_filename)
loaded_model

XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.300000012, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=6, max_leaves=0, min_child_weight=1,
             missing=nan, monotone_constraints='()', n_estimators=100, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=0, reg_alpha=0,
             reg_lambda=1, ...)

Все выглядит корректно. Можно переносить модель в приложение.